In [1]:
!pip install -q -U langchain langgraph langchain-google-genai langchain-community chromadb tiktoken sentence-transformers

In [3]:
import os
from google.colab import userdata
from typing import TypedDict, Annotated, Sequence, List
import operator
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langgraph.graph import StateGraph, END
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from pydantic import BaseModel, Field

# --- INTERACTING WITH GEMINI ---
# Import the Google GenAI modules for LangChain
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Fetch your Gemini Key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

# Initialize Gemini Embeddings and Chat Model
# Using the updated 'gemini-embedding-2' and 'gemini-2.5-flash' models
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)


# --- DOCUMENT INGESTION ---
os.makedirs('/content/data', exist_ok=True)
with open('/content/data/knowledge_base.txt', 'w') as f:
    f.write("LangGraph is a library for building stateful, multi-actor applications with LLMs. It extends LangChain to support cyclic computational steps. It is highly useful for creating reliable autonomous agents.")

loader = TextLoader('/content/data/knowledge_base.txt')
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=50)
doc_splits = text_splitter.split_documents(docs)

# Initialize ChromaDB using Gemini Embeddings
vectorstore = Chroma.from_documents(
    documents=doc_splits,
    collection_name="rag-chroma",
    embedding=embeddings,
    persist_directory="/content/chroma_db"
)
retriever = vectorstore.as_retriever()


# --- DEFINE GRAPH STATE ---
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    question: str
    documents: List[str]
    generation: str


# --- DEFINE GRAPH NODES ---
def retrieve(state: AgentState):
    print("---NODE: RETRIEVE---")
    question = state["question"]
    documents = retriever.invoke(question)
    return {"documents": documents, "question": question}
class Grade(BaseModel):
    """Binary score for relevance check."""
    score: str = Field(description="Binary score 'yes' or 'no' indicating whether the document is relevant to the question.")

def grade_documents(state: AgentState):
    print("---NODE: GRADE DOCUMENTS---")
    question = state["question"]
    documents = state["documents"]

    # Notice we removed the JSON instructions from the prompt since Pydantic handles it
    prompt = PromptTemplate(
        template="""You are a grader assessing relevance of a retrieved document to a user question. \n
        Here is the retrieved document: \n\n {document} \n\n
        Here is the user question: {question} \n
        If the document contains keywords related to the user question, grade it as relevant. \n
        Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question.""",
        input_variables=["question", "document"],
    )

    # Pass the Pydantic schema directly into the structured output method
    grader_chain = prompt | llm.with_structured_output(Grade)

    filtered_docs = []
    for d in documents:
        # The result is now automatically parsed into our Grade Python object!
        result = grader_chain.invoke({"question": question, "document": d.page_content})

        if result.score.lower() == "yes":
            filtered_docs.append(d)

    return {"documents": filtered_docs, "question": question}

def generate(state: AgentState):
    print("---NODE: GENERATE---")
    question = state["question"]
    documents = state["documents"]

    context = "\n\n".join(doc.page_content for doc in documents)

    prompt = PromptTemplate(
        template="""You are an assistant for question-answering tasks.
        Use the following pieces of retrieved context to answer the question.
        If you don't know the answer, just say that you don't know.
        Question: {question}
        Context: {context}
        Answer:""",
        input_variables=["question", "context"],
    )

    rag_chain = prompt | llm
    generation = rag_chain.invoke({"context": context, "question": question})
    return {"generation": generation.content, "question": question}


# --- CONDITIONAL ROUTING ---
def decide_to_generate(state: AgentState):
    print("---EDGE: DECIDE TO GENERATE---")
    if not state["documents"]:
        print("---DECISION: ALL DOCUMENTS ARE IRRELEVANT.---")
        return "end"
    else:
        print("---DECISION: DOCUMENTS RELEVANT. GENERATING ANSWER.---")
        return "generate"


# --- BUILD AND COMPILE GRAPH ---
workflow = StateGraph(AgentState)
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("generate", generate)

workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges(
    "grade_documents",
    decide_to_generate,
    {"generate": "generate", "end": END}
)
workflow.add_edge("generate", END)
app = workflow.compile()


# --- EXECUTE ---
inputs = {"question": "What library is used for stateful multi-actor systems?"}
for output in app.stream(inputs):
    for key, value in output.items():
        print(f"Finished step: {key}")

print("\nFinal Answer:")
print(value.get('generation', "No answer generated."))

---NODE: RETRIEVE---
Finished step: retrieve
---NODE: GRADE DOCUMENTS---
---EDGE: DECIDE TO GENERATE---
---DECISION: DOCUMENTS RELEVANT. GENERATING ANSWER.---
Finished step: grade_documents
---NODE: GENERATE---
Finished step: generate

Final Answer:
LangGraph is the library used for building stateful, multi-actor applications with LLMs.
